# Demo 3 : PDF-to-Searchable-Index Pipeline

**Module 5A - Hour 3: RAG Fundamentals & Databricks AI Search (Topics 3.1-3.10)**

This notebook demonstrates the full RAG pipeline: document processing, chunking, embeddings, AI Search index creation, and similarity queries - all on the Databricks platform.

Each section has **Notes** (concept + rationale) followed by a **Demo** cell with live code.
At the end: **Learning Conclusion** and **Cleanup** to decommission everything created.

In [0]:
%sql
-- SETUP: Create catalog, schema, and knowledge base table
-- documents table that has CDF and a primary key (both required
-- for Delta Sync vector search indexes).

CREATE CATALOG IF NOT EXISTS module5a_demo3;
CREATE SCHEMA IF NOT EXISTS module5a_demo3.rag;
CREATE VOLUME IF NOT EXISTS module5a_demo3.rag.pdf_documents;

CREATE OR REPLACE TABLE module5a_demo3.rag.knowledge_base (
  doc_id     STRING NOT NULL,
  title      STRING,
  content    STRING,
  category   STRING,
  updated_at TIMESTAMP DEFAULT current_timestamp(),
  CONSTRAINT pk_knowledge_base PRIMARY KEY (doc_id)
) TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  'delta.feature.allowColumnDefaults' = 'supported'
);



In [0]:
# Setup: Copy the sample PDF to a UC Volume
# ai_parse_document requires binary files in a UC Volume (not workspace files).
# We create the volume, then copy the PDF from the workspace static folder.

spark.sql("CREATE VOLUME IF NOT EXISTS module5a_demo3.rag.pdf_documents")

pdf_source = "file:/Workspace/Users/shriveens@platformatory.com/Data+AI Academy/Module 5A/static/databricks-ebook-a-compact-guide-to-agent-systems.pdf"
pdf_dest = "/Volumes/module5a_demo3/rag/pdf_documents/"

dbutils.fs.cp(pdf_source, pdf_dest, recurse=False)
print("PDF copied to volume")
for f in dbutils.fs.ls(pdf_dest):
    print(f"  {f.name} ({f.size:,} bytes)")

PDF copied to volume
  databricks-ebook-a-compact-guide-to-agent-systems.pdf (2,481,930 bytes)


In [0]:
%sql
-- Setup: Parse the PDF with ai_parse_document and populate the knowledge base
-- ai_parse_document(content, MAP('version', '2.0')) returns a VARIANT with:
--   parsed:document:elements - array of text, tables, figures, titles, etc.
--   parsed:document:pages - page metadata
-- We extract text elements and insert them as documents.

TRUNCATE TABLE module5a_demo3.rag.knowledge_base;

INSERT INTO module5a_demo3.rag.knowledge_base
WITH parsed AS (
  SELECT 
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '/Volumes/module5a_demo3/rag/pdf_documents/',
    format => 'binaryFile'
  )
),
elements AS (
  SELECT posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (pos, element)
  FROM parsed
  WHERE is_variant_null(parsed:error_status)
)
SELECT
  concat('doc-', lpad(cast(row_number() OVER (ORDER BY pos) AS STRING), 3, '0')) AS doc_id,
  CASE 
    WHEN element:type::STRING = 'title' THEN element:content::STRING
    WHEN element:type::STRING = 'section_header' THEN element:content::STRING
    ELSE concat('Section ', cast(pos AS STRING))
  END AS title,
  element:content::STRING AS content,
  element:type::STRING AS category,
  current_timestamp() AS updated_at
FROM elements
WHERE element:type::STRING IN ('text', 'title', 'section_header')
  AND length(coalesce(element:content::STRING, '')) > 30;

SELECT count(*) AS total_docs, collect_set(category) AS doc_types
FROM module5a_demo3.rag.knowledge_base;

total_docs,doc_types
33,"List(section_header, text)"


In [0]:
# Setup: Create a Vector Search endpoint
# This is a managed compute resource that hosts vector indexes.
# We use STANDARD for faster provisioning.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

w = WorkspaceClient()
endpoint_name = "demo3_vs_endpoint"

# Create endpoint if it doesn't exist
try:
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists: {ep.endpoint_status.state}")
except Exception:
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type=EndpointType.STANDARD,
    )
    print(f"Creating endpoint '{endpoint_name}'... (takes ~2-3 min)")

# Wait for endpoint to be ready
for i in range(30):
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
    state = str(ep.endpoint_status.state)
    print(f"  State: {state}")
    if "ONLINE" in state:
        print(f"Endpoint '{endpoint_name}' is ready!")
        break
    time.sleep(10)

Creating endpoint 'demo3_vs_endpoint'... (takes ~2-3 min)
  State: EndpointStatusState.ONLINE
Endpoint 'demo3_vs_endpoint' is ready!


## 3.1 : Context Engineering vs. Prompt Engineering

### Concepts
* Two ways to adapt a model's behavior:
  * **Prompt engineering**: craft better instructions (change what you ask).
  * **Context engineering**: provide better background knowledge (change what the model sees).
* Context engineering is the primary discipline in practice - most "prompt" issues are actually missing-context issues.
* RAG (Retrieval-Augmented Generation) is the most common context engineering pattern: retrieve relevant documents, inject them as context, then let the model answer.

`ai_query()` lets us demonstrate both approaches. Without context, the model has no knowledge of our specific platform. With context retrieved from our knowledge base, it gives grounded, specific answers.

In [0]:
%sql
-- 3.1 Demo: Context engineering vs. prompt engineering
-- Without context: the model gives a generic answer
-- With context: the model gives a grounded, specific answer

-- Without context (prompt engineering only)
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'What is an AI agent? Answer in 2 sentences.',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 100)
) AS answer_without_context;

-- With context (context engineering - we inject retrieved knowledge)
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Based on this context: "' || (
    SELECT content FROM module5a_demo3.rag.knowledge_base WHERE doc_id = 'doc-006'
  ) || '" What is an AI agent? Answer in 2 sentences.',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 100)
) AS answer_with_context;

answer_with_context
"An AI agent is an intelligent application designed to automate tasks and enhance human productivity by analyzing information, making decisions, and taking actions to achieve specific goals. AI agents can perform a variety of tasks, such as customer service, campaign generation, and code generation, to free up time and resources for more strategic work, and can be used individually or in combination with other agents to achieve their objectives."


## 3.2 : The RAG Pattern - Indexing Phase vs. Query Phase

### Concepts
RAG has two distinct phases:

* **Indexing phase** (done once or periodically):
  1. Load documents (PDFs, text, structured data)
  2. Chunk documents into smaller pieces
  3. Generate embeddings for each chunk
  4. Store embeddings in a vector search index

* **Query phase** (every user question):
  1. Embed the user's question
  2. Search the vector index for similar chunks (similarity search / ANN)
  3. Inject retrieved chunks as context into the LLM prompt
  4. LLM generates a grounded answer

We simulate the query phase: retrieve relevant context, then generate. In a full RAG pipeline, the retrieval step uses `vector_search()` against an AI Search index.

In [0]:
%sql
-- 3.2 Demo: The RAG query phase (simulated)
-- Step 1: Retrieve relevant context (here we use a direct doc lookup;
--   in production, vector_search() finds semantically similar docs)
-- Step 2: Generate a grounded answer using the retrieved context

WITH retrieved_context AS (
  SELECT content FROM module5a_demo3.rag.knowledge_base
  WHERE doc_id = 'doc-007'
  LIMIT 1
)
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Answer this question using only the provided context.\n\nContext: ' || content || '\n\nQuestion: How does an AI agent use an LLM for reasoning?',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 100)
) AS rag_answer
FROM retrieved_context;

rag_answer
"The agent uses the Large Language Model (LLM) as its ""brain"" for reasoning, implying that the LLM serves as the agent's primary mechanism for processing and generating thoughts, making decisions, and drawing conclusions."


## 3.3 : Why Retrieval Agents Exist

### Concepts
Three problems that RAG solves:
1. **Knowledge cutoff**: Models are trained on data up to a point. They don't know about events, products, or policies after that date.
2. **Hallucination**: Without grounded context, models fabricate plausible but incorrect answers.
3. **Missing private context**: Models have no access to your organization's internal documents, data, or knowledge base.

RAG addresses all three by retrieving relevant, authoritative context from your own data before generating an answer.

### When to use Feature Store vs Vector Store
Not all data belongs in a vector index:
* **Vector store / RAG**: Best for **unstructured content** (documents, policies, articles) where semantic search is needed.
* **Feature store**: Best for **structured, lookup-based data** (transaction records, pricing, delivery dates) where exact key-based lookup is needed.
* Example: A customer support app that answers shipping questions should use a **feature store table** keyed by `transaction_id` for delivery dates, NOT a vector store of shipping policies. The vector store can't return a specific expected arrival date for a specific order.

### Retrieval Gating
* **Retrieval gating** means requiring a minimum relevance score before using retrieved context.
* If the best chunk's similarity score is below a threshold, the system should **refuse to answer** ("I don't know") instead of guessing.
* This prevents hallucination from weak or irrelevant context and is critical for production RAG systems.
* Combine with a **refusal policy** in the system prompt: "If the retrieved context does not clearly answer the question, say you don't know."

In [0]:
%sql
-- 3.3 Demo: Hallucination vs. grounded answer
-- Without our knowledge base, the model may hallucinate about
-- our internal product details. With context, it stays grounded.

-- Without context: the model guesses or fabricates
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'How does Databricks Mosaic AI help build agent systems? Answer briefly.',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
) AS hallucinated_answer;

-- With context from our knowledge base: the model is grounded
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Context: "' || content || '"\n\nQuestion: How does Databricks Mosaic AI help build agent systems? Answer briefly.',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
) AS grounded_answer
FROM module5a_demo3.rag.knowledge_base
WHERE doc_id = 'doc-004';

grounded_answer
"Databricks Mosaic AI helps build agent systems by providing secure customization with enterprise data, automated tools for evaluation and improvement, and flexible connectivity to open source or commercial models."


## 3.4 : Document Processing Pipeline

### Concepts
The full document processing pipeline:
1. `ai_parse_document(content, MAP('version', '2.0'))` - parse PDFs/images into structured VARIANT (elements: text, tables, figures, titles)
2. `ai_classify(parsed_content, labels, MAP(...))` - categorize documents by type
3. `ai_extract(parsed_content, schema, MAP(...))` - extract structured fields
4. `ai_prep_search(parsed_content)` - chunk for vector search (semantic chunking)

We use a real PDF (Databricks eBook: A Compact Guide to Agent Systems) uploaded to a UC Volume. `ai_parse_document` parses it into structured VARIANT. We show the raw parsed output, then classify and extract key fields.

> `ai_parse_document` requires binary files (PDFs, images, Office docs) stored in a UC Volume. See the SQL reference for the full parsing pipeline.

### Third-Party Python Libraries for Document Processing
While `ai_parse_document` is the Databricks-native approach, the exam may test knowledge of common Python libraries:

| Library | Best For | Why |
|---|---|---|
| `pytesseract` | Scanned images (.jpeg, .png) | OCR wrapper for Tesseract engine; extracts text from images with minimal code |
| `unstructured` | Mixed-format PDFs (text + images) | Handles PDFs with both text and embedded images; least code for mixed content |
| `beautifulsoup` | HTML parsing | Parses HTML/XML documents; not for PDFs or images |
| `scrapy` | Web scraping | Crawls and extracts from web pages; not for local documents |
| `pyquery` | HTML querying | jQuery-like syntax for HTML; not for PDFs or images |
| `flask` | Web framework | Not a document processing library at all |
| `numpy` | Numerical computing | Not a document processing library |

**Exam tip**: If the source documents are **scanned images** (.jpeg, .png), use `pytesseract`. If they are **PDFs with mixed text and images**, use `unstructured`.

In [0]:
%sql
-- 3.4a Demo: Raw ai_parse_document output on the actual PDF
-- This shows what ai_parse_document returns before any processing.
-- The VARIANT contains document.pages (page metadata) and
-- document.elements (text, tables, figures, titles, etc.).

SELECT 
  path,
  size(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS num_elements,
  size(try_cast(parsed:document:pages AS ARRAY<VARIANT>)) AS num_pages,
  parsed:metadata AS metadata
FROM (
  SELECT 
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '/Volumes/module5a_demo3/rag/pdf_documents/',
    format => 'binaryFile'
  )
);

-- Show the first 10 elements with their types and a content preview
SELECT 
  element:type::STRING AS element_type,
  substring(element:content::STRING, 1, 120) AS content_preview
FROM (
  SELECT posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (pos, element)
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES(
      '/Volumes/module5a_demo3/rag/pdf_documents/',
      format => 'binaryFile'
    )
  )
  WHERE is_variant_null(parsed:error_status)
)
LIMIT 10;

element_type content_preview text Guide title A Compact Guide
to Al Agents figure figure text databricks page_header A C O M PA C T G U I D E T O A I A G E N T S page_number 2 section_header Contents table Introduction 3 What Is an AI Agent? 5 What Is an AI Age figure

In [0]:
%sql
-- 3.4 Demo: Document processing pipeline - classify and extract
-- Our knowledge_base table was populated from the real PDF using ai_parse_document.
-- Now we classify each document and extract key fields using AI Functions.

SELECT
  doc_id,
  title,
  ai_classify(
    content,
    '{"intro":"Introduction or overview of AI agents","architecture":"Agent architecture, reasoning, or design","use_cases":"Business use cases and applications","deployment":"Building, deploying, and evaluating agents","governance":"Governance, security, and compliance"}',
    MAP('version', '2.1', 'enableConfidenceScores', 'true')
  ) AS doc_category,
  ai_extract(
    content,
    '{
      "primary_topic": {"type": "string", "description": "Main subject of the document"},
      "has_code": {"type": "boolean", "description": "Does the document mention code or programming"},
      "key_terms": {"type": "array", "items": {"type": "string"}}
    }',
    MAP('version', '2.1')
  ) AS extracted_fields
FROM module5a_demo3.rag.knowledge_base
ORDER BY doc_id;

doc_id,title,doc_category,extracted_fields
doc-001,Scaling GenAI apps into production using AI agent systems,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""deployment""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""Scaling""},{""value"":""GenAI""},{""value"":""apps""},{""value"":""production""},{""value"":""AI agent systems""}],""primary_topic"":{""value"":""Scaling GenAI apps into production using AI agent systems""}}}"
doc-002,Section 15,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""deployment""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""Generative AI""},{""value"":""Enterprise AI""},{""value"":""pilot projects""},{""value"":""deployed applications""},{""value"":""scaling projects""},{""value"":""infrastructure readiness""},{""value"":""GenAI models""},{""value"":""production-ready""},{""value"":""enterprise data""},{""value"":""AI agent systems""}],""primary_topic"":{""value"":""Generative AI implementation challenges in enterprises""}}}"
doc-003,"Databricks Mosaic AI: Building high-quality, scalable agent systems","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.6,""value"":""architecture""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""Databricks""},{""value"":""Mosaic AI""},{""value"":""high-quality""},{""value"":""scalable""},{""value"":""agent systems""}],""primary_topic"":{""value"":""Building high-quality, scalable agent systems""}}}"
doc-004,Section 17,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.65,""value"":""intro""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""AI agent systems""},{""value"":""lakehouse architecture""},{""value"":""secure customization""},{""value"":""enterprise data""},{""value"":""domain-specific outputs""},{""value"":""open source models""},{""value"":""commercial models""},{""value"":""best-fit solutions""},{""value"":""automated tools""},{""value"":""agile development""},{""value"":""robust governance""},{""value"":""data to models""},{""value"":""visibility and control""}],""primary_topic"":{""value"":""Databricks Mosaic AI AI agent systems""}}}"
doc-005,Section 23,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.6,""value"":""intro""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""AI agents""},{""value"":""AI agent systems""},{""value"":""agent systems""},{""value"":""customer service automation""},{""value"":""multi-agent collaboration""},{""value"":""scalable""},{""value"":""modular agent systems""},{""value"":""enterprise data""},{""value"":""generative AI""},{""value"":""classical AI""}],""primary_topic"":{""value"":""AI agents and AI agent systems""}}}"
doc-006,Section 30,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.6,""value"":""intro""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":true},""key_terms"":[{""value"":""AI agents""},{""value"":""customer service agent""},{""value"":""campaign generation agent""},{""value"":""code generation agent""},{""value"":""AI agent system""}],""primary_topic"":{""value"":""AI agents""}}}"
doc-007,An agent uses the LLM as its brain for reasoning,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.75,""value"":""architecture""}]}","{""error_message"":null,""metadata"":{""ver

## 3.5 : Chunking Strategy Considerations

### Concepts
* **Chunk size**: Balance between too small (loses context) and too large (dilutes relevance, hits embedding model token limits).
* **Overlap**: 10-20% overlap between chunks prevents losing context at boundaries.
* **Granularity trade-off**: Smaller chunks = more precise retrieval but less context per chunk. Larger chunks = more context but less precise.
* **"Lost in the middle"**: Models pay less attention to content in the middle of long contexts. Keep chunks focused.

We demonstrate manual chunking by splitting documents into sentences using SQL string functions. In production, use `ai_prep_search` for semantic chunking that respects paragraph and section boundaries.

### Retrieval Budget and Context Compression
* **Retrieval budget**: Limit the number and size of chunks fed to the LLM based on dynamic relevance scores.
* If you retrieve 10 chunks but only have token budget for 5, keep the top-5 by relevance score.
* **Context compression**: Summarize or compress retrieved context before passing it to the generation LLM.
* This prevents **token limit errors** when the combined context + prompt exceeds the model's context window.
* Common symptom: the model truncates answers because the input context is too long.
* Mitigations: (a) reduce chunk size, (b) reduce number of retrieved records (lower k), (c) compress/summarize context before generation.

In [0]:
%sql
-- 3.5 Demo: Manual chunking of documents
-- We split documents by sentences and show each as a potential chunk.
-- In production, ai_prep_search handles this automatically with
-- semantic awareness (respecting sentence/paragraph boundaries).

WITH sentences AS (
  SELECT
    doc_id,
    title,
    sentence,
    row_number() OVER (PARTITION BY doc_id ORDER BY 1) AS sentence_num
  FROM module5a_demo3.rag.knowledge_base
  LATERAL VIEW explode(split(content, '\\. ')) AS sentence
  WHERE sentence != ''
)
SELECT
  doc_id,
  title,
  sentence_num AS chunk_id,
  length(sentence) AS chunk_length,
  sentence AS chunk_text
FROM sentences
ORDER BY doc_id, sentence_num
LIMIT 20;

doc_id,title,chunk_id,chunk_length,chunk_text
doc-001,Scaling GenAI apps into production using AI agent systems,1,57,Scaling GenAI apps into production using AI agent systems
doc-002,Section 15,1,162,"Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge"
doc-002,Section 15,2,239,"According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively"
doc-002,Section 15,3,170,"The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready"
doc-002,Section 15,4,279,"These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results"
doc-002,Section 15,5,188,"To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs."
doc-003,"Databricks Mosaic AI: Building high-quality, scalable agent systems",1,67,"Databricks Mosaic AI: Building high-quality, scalable agent systems"
doc-004,Section 17,1,93,Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems
doc-004,Section 17,2,135,"Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs"
doc-004,Section 17,3,349,"It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility and control."


## 3.6 : chunk_to_embed vs. chunk_to_retrieve

### Concepts
* **chunk_to_embed**: The text used to generate the embedding vector. Optimized for findability - may include context enrichment (title, summary, keywords) to improve semantic matching.
* **chunk_to_retrieve**: The actual text returned to the LLM as context. Clean, raw content without enrichment.
* These can be the same text, but separating them improves retrieval quality:
  * The embedding input benefits from extra context (title, category).
  * The retrieval output should be clean to avoid confusing the LLM.

We create a chunked table with both columns to show the distinction.

In [0]:
%sql
-- 3.6 Demo: chunk_to_embed vs. chunk_to_retrieve
-- We create a chunked version of our knowledge base with two text columns:
-- - chunk_to_embed: enriched with title + category for better findability
-- - chunk_to_retrieve: clean raw content for the LLM

CREATE OR REPLACE TABLE module5a_demo3.rag.document_chunks AS
WITH sentences AS (
  SELECT
    doc_id,
    title,
    category,
    sentence,
    row_number() OVER (PARTITION BY doc_id ORDER BY 1) AS sentence_num
  FROM module5a_demo3.rag.knowledge_base
  LATERAL VIEW explode(split(content, '\\. ')) AS sentence
  WHERE sentence != ''
)
SELECT
  concat(doc_id, '-', lpad(cast(sentence_num AS STRING), 3, '0')) AS chunk_id,
  doc_id,
  title,
  category,
  -- chunk_to_embed: enriched with title + category for findability
  concat(title, ' | ', category, ' | ', sentence) AS chunk_to_embed,
  -- chunk_to_retrieve: clean raw text for the LLM
  sentence AS chunk_to_retrieve
FROM sentences;

SELECT chunk_id, chunk_to_embed, chunk_to_retrieve
FROM module5a_demo3.rag.document_chunks
ORDER BY chunk_id
LIMIT 10;

chunk_id,chunk_to_embed,chunk_to_retrieve
doc-001-001,Scaling GenAI apps into production using AI agent systems | section_header | Scaling GenAI apps into production using AI agent systems,Scaling GenAI apps into production using AI agent systems
doc-002-001,"Section 15 | text | Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge","Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge"
doc-002-002,"Section 15 | text | According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively","According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively"
doc-002-003,"Section 15 | text | The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready","The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready"
doc-002-004,"Section 15 | text | These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results","These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results"
doc-002-005,"Section 15 | text | To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs.","To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs."
doc-003-001,"Databricks Mosaic AI: Building high-quality, scalable agent systems | section_header | Databricks Mosaic AI: Building high-quality, scalable agent systems","Databricks Mosaic AI: Building high-quality, scalable agent systems"
doc-004-001,Section 17 | text | Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems,Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems
doc-004-002,"Section 17 | text | Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs","Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs"
doc-004-003,"Section 17 | text | It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility and control.","It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility 

## 3.7 : Embeddings & Cosine Similarity

### Concepts
* **Embeddings**: Numeric vectors that capture the semantic meaning of text. Similar meanings = similar vectors.
* **Cosine similarity**: Measures the angle between two vectors. 1 = identical meaning, 0 = unrelated.
* **ANN (Approximate Nearest Neighbor)**: The algorithm AI Search uses to find similar vectors efficiently without comparing against all vectors.
* Databricks provides managed embedding models: `databricks-gte-large-en` (1024 dims, 8192 token context) and `databricks-bge-large-en` (1024 dims, 512 token context).

### Embedding Model Selection: Size, Dimensions, and Context Length
When choosing an embedding model, balance **cost/latency** vs **quality**:

| Model | Context Length | Embedding Dimensions | Model Size | Best For |
|---|---|---|---|---|
| Small (e.g., MiniLM) | 512 | 384 | ~0.13 GB | Cost-sensitive, low-latency, short chunks |
| Medium (e.g., BGE Small) | 514 | 768 | ~0.44 GB | Balanced cost/quality |
| Large (e.g., BGE Large) | 2048 | 2560 | ~11 GB | Higher quality, longer context |
| XLarge (e.g., GTE Large) | 32768 | 4096 | ~14 GB | Maximum quality, long documents |

* **Cost/latency priority**: Choose the smallest model whose context length covers your chunk size. If chunks are 512 tokens, a 0.13 GB model with 384 dims is sufficient.
* **Quality priority**: Choose larger models with more dimensions for better semantic discrimination.

### Embedding Model Versioning and Dimension Contracts
* **Vector dimensions are fixed** by the embedding model. You cannot mix vectors of different dimensions in the same index.
* When you change the embedding model (e.g., from 384-dim to 1024-dim), you must **rebuild the entire index** — existing embeddings are incompatible.
* **Never zero-pad** vectors to match a new dimension. This produces garbage vectors that destroy retrieval quality.
* Best practice: **Enforce a contract** — store the model name and dimension alongside the vectors. Reject any mismatched embeddings. Rebuild the index when the model changes.
* Schema changes in source tables can cause silent dimension mismatches if the embedding model is swapped without updating the index.

We use the Databricks SDK to call an embedding model and compute cosine similarity between two texts.

In [0]:
# 3.7 Demo: Generate embeddings and compute cosine similarity
# We call the embedding model endpoint to generate vectors for two texts
# and compute cosine similarity between them.

import mlflow.deployments
import numpy as np

client = mlflow.deployments.get_deploy_client("databricks")

# Generate embeddings for two texts
response1 = client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": ["AI agents use LLMs as their brain for reasoning and decision-making."]}
)
response2 = client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": ["Delta Lake provides ACID transactions for reliable data storage."]}
)

emb1 = response1["data"][0]["embedding"]
emb2 = response2["data"][0]["embedding"]

print(f"Embedding dimensions: {len(emb1)}")
print(f"First 5 values (text 1): {[round(v, 4) for v in emb1[:5]]}")
print(f"First 5 values (text 2): {[round(v, 4) for v in emb2[:5]]}")

# Cosine similarity
cos_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
print(f"\nCosine similarity: {cos_sim:.4f}")
print(f"(1.0 = identical meaning, 0.0 = unrelated)")

Embedding dimensions: 1024
First 5 values (text 1): [-0.1012, 0.245, -0.1185, 0.5103, 0.2091]
First 5 values (text 2): [0.0606, -1.0908, -0.0645, 0.5757, 0.1171]

Cosine similarity: 0.6020
(1.0 = identical meaning, 0.0 = unrelated)


## 3.8 : Creating & Syncing an AI Search Index

### Concepts
* **Endpoint**: Compute resource that hosts indexes (Standard or Storage-Optimized).
* **Delta Sync Index**: Automatically syncs with a source Delta table. Databricks computes embeddings for you.
* **Key parameters**:
  * `endpoint_name`: Vector search endpoint to host the index
  * `source_table_name`: Delta table with CDF and primary key
  * `embedding_model_endpoint_name`: Model for computing embeddings (e.g., `databricks-gte-large-en`)
  * `pipeline_type`: `TRIGGERED` (manual sync) or `CONTINUOUS` (auto-sync)
* **Change Data Feed (CDF)**: Required for Delta Sync - enables incremental updates.
* **Primary Key**: Required constraint on the source table.

We created the endpoint in setup. Now we create the index, sync it, and query it.

### Standard vs Storage-Optimized Endpoints
* **Standard**: Faster provisioning, good for smaller indexes (< ~25M vectors). Higher query throughput.
* **Storage-Optimized**: Designed for **large-scale indexes** (100M+ vectors). Lower storage cost per vector, slightly higher latency but handles massive inventories.
* Rule of thumb: 100M+ items or latency-sensitive at scale -> Storage-Optimized.

### Hybrid Search and Reranking
* **Hybrid search** combines **vector similarity** (semantic) with **lexical search** (BM25 / keyword matching) to improve retrieval quality.
* **BM25 (Best Matching 25)**: A ranking function that scores documents by keyword frequency and inverse document frequency. It excels at exact-term matching (product names, SKUs, IDs).
* **Why combine?**: Vector search catches semantic similarity ("AI agent" matches "intelligent assistant"). BM25 catches exact terms ("ORD-001" matches "ORD-001"). Together they cover both.
* **Reranking**: A second-pass model that re-scores the top-k results from hybrid search to produce a more accurate final ranking.
* **When to use**: Turn on hybrid search + reranking when accuracy matters and you can afford slightly higher latency. Turn off for maximum speed.
* For 100M items where latency is critical: use **Storage-Optimized** + GTE Large + hybrid search + reranking — the storage-optimized endpoint keeps latency acceptable at scale.

### Vector Search Data Format
* Vector Search requires **one row per chunk** with a **unique primary key**.
* If your dataframe has an array of chunks per document (one row, multiple chunks), you must **flatten** it: one chunk per row, each with its own unique ID.
* Do NOT store chunks as individual JSON files in UC volumes — Delta tables with one row per chunk are the performant format for indexing.

In [0]:
# 3.8 Demo: Create AI Search index, sync, and query
# We create a Delta Sync index from our knowledge_base table.
# Databricks automatically computes embeddings using databricks-gte-large-en.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn,
    VectorIndexType, PipelineType
)
import time

w = WorkspaceClient()

index_name = "module5a_demo3.rag.knowledge_base_index"
endpoint_name = "demo3_vs_endpoint"

# Create the index (delete and recreate if not ready)
try:
    idx = w.vector_search_indexes.get_index(index_name=index_name)
    if idx.status.ready:
        print(f"Index already exists and is ready!")
    else:
        print("Index not ready. Deleting and recreating...")
        w.vector_search_indexes.delete_index(index_name=index_name)
        time.sleep(15)
        raise Exception("Force recreate")
except Exception:
    print(f"Creating index '{index_name}'...")
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="doc_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="module5a_demo3.rag.knowledge_base",
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="content",
                    embedding_model_endpoint_name="databricks-gte-large-en"
                )
            ],
            pipeline_type=PipelineType.TRIGGERED,
            columns_to_sync=["doc_id", "title", "content", "category"],
        ),
    )
    print("Index creation submitted.")

# Wait for index to be ready
for i in range(60):
    idx = w.vector_search_indexes.get_index(index_name=index_name)
    print(f"  Index ready: {idx.status.ready}")
    if idx.status.ready:
        print("Index is ready!")
        break
    time.sleep(10)

# Sync the index (TRIGGERED mode requires manual sync)
print("\nSyncing index...")
try:
    w.vector_search_indexes.sync_index(index_name=index_name)
    time.sleep(5)
except Exception as e:
    print(f"  Sync already in progress, waiting...")
    time.sleep(30)

# Query the index - semantic search
print("\n=== Semantic Search: 'What is an AI agent?' ===")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text="What is an AI agent?",
    num_results=3,
)

for row in (results.result.data_array if results.result and results.result.data_array else []):
    score = row[-1]
    print(f"  Score: {score:.4f} | {row[1]} | {row[2][:80]}...")

print("\n=== Semantic Search: 'How do you build an agent system?' ===")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text="How do you build an agent system?",
    num_results=3,
)

for row in (results.result.data_array if results.result and results.result.data_array else []):
    score = row[-1]
    print(f"  Score: {score:.4f} | {row[1]} | {row[2][:80]}...")

Index already exists and is ready!
  Index ready: True
Index is ready!

Syncing index...

=== Semantic Search: 'What is an AI agent?' ===
  Score: 0.6878 | Section 30 | AI agents are intelligent applications designed to automate tasks and enhance hu...
  Score: 0.6676 | What Are Use
Cases for an AI
Agent System? | What Are Use
Cases for an AI
Agent System?...
  Score: 0.6459 | AI agent system applications across industries | AI agent system applications across industries...

=== Semantic Search: 'How do you build an agent system?' ===
  Score: 0.7247 | What’s the Next
Step If I Want to
Start Building AI
Agent Systems? | What’s the Next
Step If I Want to
Start Building AI
Agent Systems?...
  Score: 0.6681 | Section 85 | Creating an effective AI agent system requires a structured approach to ensure i...
  Score: 0.6591 | Build high-quality AI agent systems — see how | Build high-quality AI agent systems — see how...


## 3.9 : Managed RAG (Knowledge Assistant) vs. Custom RAG (Agent Framework + AI Search)

### Concepts

| Dimension | Managed RAG (Knowledge Assistant) | Custom RAG (Agent Framework + AI Search) |
|---|---|---|
| Setup time | Minutes (no-code UI) | Hours to days (code) |
| Customizability | Limited (pre-built) | Full control |
| Chunking | Automatic | You control strategy |
| Retrieval | Automatic | You tune search parameters |
| Model choice | Limited to configured models | Any model via endpoints |
| When to use | Quick POC, simple Q&A | Production, complex requirements |

* **Knowledge Assistant** (Agent Bricks): point at a UC volume of documents, get a chatbot. Best for fast time-to-value.
* **Custom RAG**: build your own pipeline with Agent Framework + AI Search. Best when you need custom chunking, filtering, or multi-step retrieval.

This demo is a **Custom RAG** pipeline: we controlled the table schema, chunking, embedding model, and index parameters.

## 3.10 : Unity Catalog Governance Across the Retrieval Pipeline

### Concepts
Every component of the RAG pipeline is governed by Unity Catalog:
* **Volumes**: Source documents stored in UC Volumes (governed storage)
* **Tables**: Parsed/chunked documents are Delta tables (table-level ACLs)
* **Indexes**: AI Search indexes are UC entities (create/query permissions)
* **Endpoints**: Vector Search endpoints have separate ACLs (CAN_USE)
* **Models**: Embedding and LLM endpoints have their own access controls
* **Lineage**: UC tracks data flow from source documents through chunks to index to answer

All permissions are managed in one place - no separate governance system for AI components.

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| 3.1 | `ai_query` with and without context | Context engineering > prompt engineering for grounded answers |
| 3.2 | Manual RAG query (retrieve + generate) | RAG = indexing phase + query phase |
| 3.3 | Hallucinated vs. grounded answer | RAG prevents hallucination; feature store for structured data, vector store for unstructured; retrieval gating with relevance threshold |
| 3.4 | `ai_classify` + `ai_extract` on documents | Document pipeline: parse, classify, extract; pytesseract for images, `unstructured` for mixed PDFs |
| 3.5 | Manual sentence-level chunking | Chunk size/overlap affect quality; retrieval budget limits context fed to LLM |
| 3.6 | `chunk_to_embed` vs `chunk_to_retrieve` columns | Enriched text for findability, clean text for the LLM |
| 3.7 | Embedding model + cosine similarity | Embeddings capture meaning; model selection by size/dims/context; versioning requires index rebuild |
| 3.8 | Vector Search endpoint + Delta Sync index + query | Standard vs Storage-Optimized; hybrid search + BM25 + reranking; one chunk per row |
| 3.9 | Managed vs. Custom RAG comparison | Knowledge Assistant for speed, Custom RAG for control |
| 3.10 | UC governance overview | Every RAG component governed by Unity Catalog |

### Key principles
* **RAG = retrieve + generate**: Get relevant context from your data, then let the LLM answer.
* **Chunking matters**: How you split documents affects retrieval quality.
* **AI Search is managed**: Databricks computes embeddings, maintains the index, and serves queries.
* **Everything is governed**: Unity Catalog controls access to every component.
* **Start managed, go custom when needed**: Knowledge Assistant first, Custom RAG when you need control.

In [0]:
# CLEANUP: Decommission everything created in this demo
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

index_name = "module5a_demo3.rag.knowledge_base_index"
endpoint_name = "demo3_vs_endpoint"

# 1. Delete the vector search index
print("Deleting index...")
try:
    w.vector_search_indexes.delete_index(index_name=index_name)
    print(f"  Deleted index: {index_name}")
except Exception as e:
    print(f"  Index deletion: {e}")

time.sleep(3)

# 2. Delete the vector search endpoint
print("Deleting endpoint...")
try:
    w.vector_search_endpoints.delete_endpoint(endpoint_name=endpoint_name)
    print(f"  Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"  Endpoint deletion: {e}")

# 3. Drop the schema and catalog (cascades to tables)
print("Dropping schema and catalog...")
spark.sql("DROP VOLUME IF EXISTS module5a_demo3.rag.pdf_documents")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo3.rag CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo3 CASCADE")
print("  Dropped schema and catalog")

print("\nCleanup complete!")

Deleting index...
  Deleted index: module5a_demo3.rag.knowledge_base_index
Deleting endpoint...
  Deleted endpoint: demo3_vs_endpoint
Dropping schema and catalog...
  Dropped schema and catalog

Cleanup complete!
